# Feature Selection

### Abstract

This notebook identifies the optimal predictor subset for the geomagnetic storm forecast models developed in `cosmic_ray_storm_prediction.ipynb`. Feature selection is performed via three independent methods — weighted LASSO, Mutual Information, and SHAP-based importance — applied across all five forecast horizons (1, 3, 7, 12, 21 hours). A feature is retained in `SELECTED_FEATURES` if it survives in at least two of three methods at any horizon (majority vote, union across horizons).

Storm hours (Dst < −50 nT, ~5% of the training set) are upweighted by the inverse storm fraction in all three methods to prevent suppression of Forbush Decrease signals concentrated in storm periods `[KIS25]`.

The primary output is `models/selected_features.pkl` — a dictionary mapping each forecast horizon to its optimal feature subset, consumed by the modelling notebook.

---
## Table of Contents

1. [Setup](#setup)
2. [Data Loading](#data-loading)
3. [Feature Selection Pipeline](#pipeline)
   - 3.1 [Storm Weights](#storm-weights)
   - 3.2 [LASSO Screening](#lasso)
   - 3.3 [Mutual Information](#mi)
   - 3.4 [SHAP Importance](#shap)
   - 3.5 [Majority Vote Consolidation](#consolidation)
4. [Results](#results)
   - 4.1 [Summary Table — Features × Methods × Horizons](#summary-table)
   - 4.2 [Selected Features per Horizon](#selected-per-horizon)
   - 4.3 [d_neutron Analysis — H2 Diagnostic](#h2)
5. [Save Results](#save)

---

In [1]:
# ── 1. Setup ──────────────────────────────────────────────────────────────────
import sys
sys.path.insert(0, '..') 

import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings('ignore')

from src.feature_selector import LassoSelector, MISelector, SHAPSelector
from src.selection_pipeline import FeatureSelectionPipeline
from src.utils import build_storm_weights

print("Imports OK")
print(f"  numpy  : {np.__version__}")
print(f"  pandas : {pd.__version__}")

import sklearn
import shap
print(f"  sklearn: {sklearn.__version__}")
print(f"  shap   : {shap.__version__}")

Imports OK
  numpy  : 1.26.4
  pandas : 2.3.0+4.g1dfc98e16a
  sklearn: 1.6.1
  shap   : 0.49.1


In [2]:
# ── 2. Data Loading ───────────────────────────────────────────────────────────
feat  = pd.read_parquet('../data/processed/feat_split.parquet')
masks = joblib.load('../models/split_masks.pkl')
ctx   = joblib.load('../models/context_constants.pkl')

FEATURE_COLS = ctx['FEATURE_COLS']
K_HORIZONS   = ctx['K_HORIZONS']
STORM_THR    = ctx['STORM_THR']

# Train segment only — feature selection uses training data exclusively
X_train = feat.loc[masks['train'], FEATURE_COLS]

print(f"feat shape     : {feat.shape}")
print(f"FEATURE_COLS   : {len(FEATURE_COLS)} features")
print(f"K_HORIZONS     : {K_HORIZONS}")
print(f"X_train shape  : {X_train.shape}")
print(f"\nTrain date range: {feat.loc[masks['train'], 'datetime'].min()} → "
      f"{feat.loc[masks['train'], 'datetime'].max()}")

feat shape     : (364728, 73)
FEATURE_COLS   : 33 features
K_HORIZONS     : [1, 3, 7, 12, 21]
X_train shape  : (121185, 33)

Train date range: 1995-01-01 00:00:00 → 2008-12-31 02:00:00


> **Observations — Data Loading:**
> - `feat_split.parquet`: 364,728 rows × 73 columns — scaled and imputed, all segments included.
> - `X_train`: 121,185 rows × 33 features — Train\_1 ∪ Train\_2, purge zones excluded.
> - Train period: 1995-01-01 → 2008-12-31 (Solar Cycles 22–23, with Val\_Storm carved out).
> - `K_HORIZONS = [1, 3, 7, 12, 21]` — five forecast horizons covering sub-τ to Forbush lead time range.

<a id='pipeline'></a>
## 3. Feature Selection Pipeline

Three independent feature selection methods are applied across all five forecast horizons. A feature is retained in `SELECTED_FEATURES` if it survives in at least **2 of 3 methods** at any horizon (majority vote, union across horizons).

| Method | Type | Motivation |
|---|---|---|
| `LassoSelector` | Linear | L1 regularisation — identifies features with non-zero linear contribution at cross-validated penalty level |
| `MISelector` | Non-linear | Mutual Information — captures dependencies LASSO misses by definition |
| `SHAPSelector` | Non-linear | SHAP on lightweight RF — preserves temporal structure of lag features; stable under multicollinearity (VIF 159–169) |

All three methods use **storm sample weights** (w = 1/storm\_fraction ≈ 21×) for dst < −50 nT — motivated by the concentration of Forbush Decrease signals in storm periods `[KIS25]`.

The union-across-horizons criterion ensures that `d_neutron` — which carries its primary signal at h = 12–21 h rather than h = 1 h — is not excluded by screening performed at a single horizon.

In [3]:
pipeline_strict = FeatureSelectionPipeline(
    feature_cols = FEATURE_COLS,
    k_horizons   = K_HORIZONS,
    storm_thr    = STORM_THR,
    lasso_splits = 5,
    rfecv_splits = 5,
    min_votes    = 2,
    random_state = 42,
)
pipeline_strict.fit(X_train, feat)

pipeline_liberal = FeatureSelectionPipeline(
    feature_cols = FEATURE_COLS,
    k_horizons   = K_HORIZONS,
    storm_thr    = STORM_THR,
    lasso_splits = 5,
    rfecv_splits = 5,
    min_votes    = 1,
    random_state = 42,
)
pipeline_liberal.fit(X_train, feat)

print("\n── Strict (intersection) ──────────────────────────────")
pipeline_strict.print_summary()

print("\n── Liberal (union) ────────────────────────────────────")
pipeline_liberal.print_summary()

print(f"\nStrict : {len(pipeline_strict.get_selected())} features")
print(f"Liberal: {len(pipeline_liberal.get_selected())} features")
print(f"Difference: {set(pipeline_liberal.get_selected()) - set(pipeline_strict.get_selected())}")


── Horizon 1h ──────────────────────────────────
  [1/2] LassoCV...
  storm fraction: 0.0471
  weight max: 21.2
  y_fit range: -387.0 to 67.0
        alpha=1.641248  retained=17/33
  [2/2] RFECV...
        retained=32/33

── Horizon 3h ──────────────────────────────────
  [1/2] LassoCV...
  storm fraction: 0.0471
  weight max: 21.2
  y_fit range: -387.0 to 67.0
        alpha=0.468026  retained=21/33
  [2/2] RFECV...
        retained=33/33

── Horizon 7h ──────────────────────────────────
  [1/2] LassoCV...
  storm fraction: 0.0471
  weight max: 21.2
  y_fit range: -387.0 to 67.0
        alpha=0.311265  retained=20/33
  [2/2] RFECV...
        retained=29/33

── Horizon 12h ──────────────────────────────────
  [1/2] LassoCV...
  storm fraction: 0.0471
  weight max: 21.2
  y_fit range: -387.0 to 67.0
        alpha=0.199308  retained=22/33
  [2/2] RFECV...
        retained=28/33

── Horizon 21h ──────────────────────────────────
  [1/2] LassoCV...
  storm fraction: 0.0471
  weight max: 21

In [5]:
# ── 6. Save Results ───────────────────────────────────────────────────────────

results = {
    'selected_strict'  : pipeline_strict.get_selected(),
    'selected_liberal' : pipeline_liberal.get_selected(),
    'summary_strict'   : pipeline_strict.summary_,
    'summary_liberal'  : pipeline_liberal.summary_,
    'vote_counts_strict' : pipeline_strict.vote_counts_,
    'vote_counts_liberal': pipeline_liberal.vote_counts_,
    'results_strict'   : pipeline_strict.results_,
    'results_liberal'  : pipeline_liberal.results_,
}

joblib.dump(results, '../models/feature_selection_results.pkl')
print("Saved: ../models/feature_selection_results.pkl")
print(f"\nStrict  : {len(results['selected_strict'])} features")
print(f"Liberal : {len(results['selected_liberal'])} features")
print(f"\nSelected (strict):")
for f in results['selected_strict']:
    print(f"  {f}")

Saved: ../models/feature_selection_results.pkl

Strict  : 28 features
Liberal : 33 features

Selected (strict):
  bz_gsm
  sw_speed
  sw_density
  sw_pressure
  sw_temp
  e_field
  plasma_beta
  mach_alfven
  f107
  ssn
  neutron_counts
  d_neutron
  bz_acc_3h
  bz_acc_6h
  bz_acc_12h
  bz_gsm_lag1
  bz_gsm_lag3
  bz_gsm_lag12
  bz_gsm_lag21
  sw_speed_lag1
  sw_speed_lag3
  sw_speed_lag7
  sw_speed_lag12
  neutron_counts_lag3
  neutron_counts_lag7
  neutron_counts_lag12
  solar_sin
  solar_cos


In [6]:
# Pearson correlation на SELECTED_FEATURES с dst_target_7h
selected = pipeline_strict.get_selected()
corr = feat[selected + ['dst_target_7h']].corr()['dst_target_7h'].drop('dst_target_7h')
print(corr.abs().sort_values(ascending=False).to_string())

e_field                 0.299122
sw_pressure             0.285280
bz_gsm                  0.276766
bz_gsm_lag1             0.258622
bz_acc_3h               0.227407
bz_gsm_lag3             0.223423
f107                    0.205513
ssn                     0.201554
bz_acc_6h               0.192914
sw_temp                 0.189636
sw_speed                0.173274
sw_speed_lag1           0.173043
sw_speed_lag3           0.171915
sw_speed_lag7           0.169209
sw_speed_lag12          0.166333
bz_acc_12h              0.161330
neutron_counts_lag12    0.142918
neutron_counts_lag7     0.142642
neutron_counts_lag3     0.142519
neutron_counts          0.142491
bz_gsm_lag12            0.133835
solar_cos               0.133668
bz_gsm_lag21            0.100663
plasma_beta             0.099967
sw_density              0.060480
solar_sin               0.033798
mach_alfven             0.002501
d_neutron               0.001986
